## পরিচিতি 

এই পাঠে আলোচনা করা হবে: 
- ফাংশন কলিং কী এবং এর ব্যবহার ক্ষেত্রসমূহ 
- কিভাবে Azure OpenAI ব্যবহার করে ফাংশন কল তৈরি করবেন 
- কিভাবে একটি অ্যাপে ফাংশন কল ইন্টিগ্রেট করবেন 

## শেখার লক্ষ্যসমূহ 

এই পাঠ সম্পন্ন করার পরে আপনি জানবেন কিভাবে এবং বুঝতে পারবেন: 

- ফাংশন কলিং ব্যবহারের উদ্দেশ্য 
- Azure OpenAI সার্ভিস ব্যবহার করে ফাংশন কল সেটআপ 
- আপনার অ্যাপ্লিকেশনের ব্যবহারের ক্ষেত্রে উপযোগী কার্যকর ফাংশন কল ডিজাইন করা 


## ফাংশন কল বোঝা

এই পাঠের জন্য, আমরা আমাদের শিক্ষা স্টার্টআপের জন্য একটি ফিচার তৈরি করতে চাই যা ব্যবহারকারীদের একটি চ্যাটবট ব্যবহার করে প্রযুক্তিগত কোর্স খুঁজে পাওয়ার সুযোগ দেয়। আমরা তাদের দক্ষতা স্তর, বর্তমান ভূমিকা এবং আগ্রহের প্রযুক্তি অনুসারে কোর্স সুপারিশ করব।

এটি সম্পূর্ণ করতে আমরা একটি সংমিশ্রণ ব্যবহার করব:
 - ব্যবহারকারীর জন্য একটি চ্যাট অভিজ্ঞতা তৈরি করতে `Azure Open AI`
 - ব্যবহারকারীর অনুরোধের ভিত্তিতে কোর্স খুঁজে পেতে সাহায্য করার জন্য `Microsoft Learn Catalog API`
 - ব্যবহারকারীর প্রশ্ন নিয়ে API অনুরোধ তৈরি করার জন্য `Function Calling`

শুরু করার জন্য, চলুন দেখা যাক কেন আমরা প্রথম থেকেই ফাংশন কল ব্যবহার করতে চাই:


### কেন ফাংশন কলিং 

যদি আপনি এই কোর্সের অন্য কোনো লেসন সম্পন্ন করে থাকেন, তাহলে সম্ভবত আপনি লার্জ ল্যাঙ্গুয়েজ মডেলস (LLMs) ব্যবহারের শক্তি বুঝতে পেরেছেন। আশা করি আপনি তাদের কিছু সীমাবদ্ধতাও দেখতে পেয়েছেন। 

ফাংশন কলিং হল Azure Open AI সার্ভিসের একটি বৈশিষ্ট্য যা নিম্নলিখিত সীমাবদ্ধতাগুলো কাটিয়ে উঠতে সাহায্য করে: 
১) সুষম প্রতিক্রিয়া ফরম্যাট 
২) একটি চ্যাট প্রসঙ্গে অ্যাপ্লিকেশনের অন্যান্য সোর্স থেকে ডেটা ব্যবহার করার ক্ষমতা 

ফাংশন কলিং এর আগে, LLM থেকে প্রতিক্রিয়াগুলো ছিল অসংগঠিত এবং অস্বচ্ছ। ডেভেলপারদের প্রতিটি ভিন্ন ধরনের প্রতিক্রিয়া হ্যান্ডেল করার জন্য জটিল ভ্যালিডেশন কোড লিখতে হতো। 

ব্যবহারকারীরা "স্টকহোমে বর্তমানে আবহাওয়া কেমন?" এর মতো প্রশ্নের উত্তর পেতেন না। কারণ মডেলগুলো প্রশিক্ষিত ডেটার সময় পর্যন্ত সীমাবদ্ধ ছিল। 

নিচের উদাহরণটি এই সমস্যাটি বিশ্লেষণ করে: 

ধরুন আমরা ছাত্রদের ডেটার একটি ডাটাবেস তৈরি করতে চাই যাতে তাদের সঠিক কোর্স সাজেস্ট করতে পারি। নিচে আমাদের কাছে দুইজন ছাত্রের বর্ণনা আছে যাদের ডেটা খুবই একইরকম। 


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


আমরা এটি একটি LLM-এ পাঠাতে চাই যাতে এটি ডেটা পার্স করতে পারে। পরে আমাদের অ্যাপে এটি একটি API-তে পাঠানোর জন্য অথবা একটি ডাটাবেজে সংরক্ষণ করার জন্য ব্যবহার করা যাবে।

আমরা দুটি একই রকম প্রম্পট তৈরি করব যেখানে আমরা LLM-কে নির্দেশ দেব কোন তথ্যের প্রতি আমরা আগ্রহী:


আমরা এটি একটি LLM-এ পাঠাতে চাই যাতে সে আমাদের পণ্যের জন্য গুরুত্বপূর্ণ অংশগুলো বিশ্লেষণ করতে পারে। তাই আমরা LLM-কে নির্দেশ দিতে দুইটি একই রকম প্রাম্পট তৈরি করতে পারি: 


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


এই দুইটি প্রম্পট তৈরি করার পরে, আমরা `client.responses.create` ব্যবহার করে সেগুলো LLM-এ পাঠাবো। আমরা প্রম্পটটি `input` ভেরিয়েবলে সংরক্ষণ করি এবং রোলটি `user` এ ধার্য করি। এটি একটি ব্যবহারকারীর বার্তা চ্যাটবটকে লেখা হচ্ছে এমন ভাব প্রকাশ করতে। 



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

# The OpenAI client points at the Azure OpenAI (Microsoft Foundry) /openai/v1/ endpoint
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']


: 

এখন আমরা উভয় অনুরোধই LLM-এ পাঠাতে পারি এবং আমরা যে প্রতিক্রিয়া পাই তা পরীক্ষা করতে পারি। 


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



যদিও প্রম্পটগুলি একই এবং বর্ণনাগুলি অনুরূপ, আমরা `Grades` সম্পত্তির বিভিন্ন ফরম্যাট পেতে পারি।

যদি আপনি উপরের সেলটি একাধিকবার চালান, ফরম্যাট হতে পারে `3.7` বা `3.7 GPA`।

এটি কারণ LLM লেখিত প্রম্পটের আকারে অস্ট্রাকচার্ড ডেটা গ্রহণ করে এবং অস্ট্রাকচার্ড ডেটা ফেরত দেয়। আমাদের একটি কাঠামোবদ্ধ ফরম্যাট থাকা দরকার যাতে আমরা যখন এই ডেটা সংরক্ষণ বা ব্যবহার করি তখন আমরা কী আশা করব তা বুঝতে পারি।

ফাংশনাল কলিং ব্যবহার করে, আমরা নিশ্চিত করতে পারি যে আমরা কাঠামোবদ্ধ ডেটা পেয়ে থাকি। ফাংশন কলিং ব্যবহার করার সময়, LLM প্রকৃতপক্ষে কোনও ফাংশন কল করে বা চালায় না। পরিবর্তে, আমরা LLM এর প্রতিক্রিয়ার জন্য একটি কাঠামো তৈরি করি যা এটি অনুসরণ করবে। এরপর আমরা আমাদের অ্যাপ্লিকেশনগুলিতে কোন ফাংশনটি চালাতে হবে তা জানা জন্য সেই কাঠামোবদ্ধ প্রতিক্রিয়াগুলি ব্যবহার করি।
 


![ফাংশন কল ফ্লো ডায়াগ্রাম](../../../../translated_images/bn/Function-Flow.083875364af4f4bb.webp)


অনতঃপর আমরা ফাংশন থেকে যা প্রত্যাবর্তিত হবে তা গ্রহণ করে আবার LLM-এ পাঠাতে পারি। LLM তখন ব্যবহারকারীর প্রশ্নের উত্তর দেওয়ার জন্য স্বাভাবিক ভাষা ব্যবহার করবে। 


### ফাংশন কল ব্যবহারের ব্যবহারিক ক্ষেত্রসমূহ  

**বাহ্যিক টুল কল করা**  
চ্যাটবটগুলো ব্যবহারকারীদের প্রশ্নের উত্তর দেওয়ার ক্ষেত্রে চমৎকার। ফাংশন কলিং ব্যবহার করে, চ্যাটবটগুলো ব্যবহারকারীদের মেসেজ ব্যবহার করে নির্দিষ্ট কাজ সম্পন্ন করতে পারে। উদাহরণস্বরূপ, একজন ছাত্র চ্যাটবটকে বলতে পারে "আমার ইন্সট্রাক্টরকে একটি ইমেইল পাঠাও যে আমি এই বিষয়ে আরো সহায়তা চাই"। এটি `send_email(to: string, body: string)` নামক ফাংশন কল করতে পারে  


**এপিআই বা ডাটাবেস কোয়েরি তৈরি করা**  
ব্যবহারকারীরা প্রাকৃতিক ভাষা ব্যবহার করে তথ্য খুঁজে পেতে পারে যা একটি ফর্ম্যাটেড কোয়েরি বা এপিআই রিকোয়েস্টে রূপান্তরিত হয়। এর একটি উদাহরণ হতে পারে একজন শিক্ষক যিনি জানতে চান "সেই ছাত্ররা কে যারা শেষ অ্যাসাইনমেন্ট সম্পন্ন করেছে" যেটা `get_completed(student_name: string, assignment: int, current_status: string)` নামক একটি ফাংশন কল করতে পারে  


**গঠনমূলক তথ্য তৈরি করা**  
ব্যবহারকারীরা একটি টেক্সট ব্লক বা CSV গ্রহণ করে LLM ব্যবহার করে তৎস্থল থেকে গুরুত্বপূর্ণ তথ্য নির্গত করতে পারে। উদাহরণস্বরূপ, একজন ছাত্র একটি উইকিপিডিয়া আর্টিকেল যেটি শান্তি চুক্তি সম্পর্কে সেটি AI ফ্ল্যাশ কার্ড তৈরি করার জন্য রূপান্তর করতে পারে। এটি করা যেতে পারে `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)` নামক ফাংশন ব্যবহার করে  


## 2. আপনার প্রথম ফাংশন কল তৈরি করা 

একটি ফাংশন কল তৈরি করার প্রক্রিয়ায় ৩টি প্রধান ধাপ রয়েছে: 
১. আপনার ফাংশনগুলির একটি তালিকা এবং একটি ব্যবহারকারী ম্যাসেজ সহ Chat Completions API কল করা 
২. একটি ক্রিয়া সম্পাদনের জন্য মডেলের প্রতিক্রিয়া পড়া, অর্থাৎ একটি ফাংশন বা API কল কার্যকর করা 
৩. ব্যবহারকারীর জন্য একটি প্রতিক্রিয়া তৈরি করার জন্য আপনার ফাংশনের প্রতিক্রিয়ার সাথে Chat Completions API-তে আরেকটি কল করা। 


![একটি ফাংশন কলের প্রবাহ](../../../../translated_images/bn/LLM-Flow.3285ed8caf4796d7.webp)


### একটি ফাংশন কলের উপাদানগুলি 

#### ব্যবহারকারীর ইনপুট 

প্রথম ধাপে একটি ব্যবহারকারী বার্তা তৈরি করতে হয়। এটি একটি টেক্সট ইনপুটের মান গ্রহণ করে গতিশীলভাবে নির্ধারিত হতে পারে বা আপনি এখানে একটি মান বরাদ্দ করতে পারেন। যদি এটি আপনার প্রথমবারের মত Chat Completions API এর সাথে কাজ করা হয়, তাহলে আমাদের মেসেজের `role` এবং `content` সংজ্ঞায়িত করতে হবে। 

`role` হতে পারে `system` (নিয়ম তৈরি করা), `assistant` (মডেল) অথবা `user` (অন্তিম ব্যবহারকারী)। ফাংশন কলিং এর জন্য, আমরা এটিকে `user` এবং একটি উদাহরণ প্রশ্ন হিসেবে বরাদ্দ করব। 


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### ফাংশন তৈরি করা। 

পরবর্তীতে আমরা একটি ফাংশন এবং সেই ফাংশনের প্যারামিটারগুলো সংজ্ঞায়িত করব। এখানে আমরা শুধুমাত্র একটি ফাংশন ব্যবহার করব যার নাম `search_courses` কিন্তু আপনি একাধিক ফাংশন তৈরি করতে পারেন।

**গুরুত্বপূর্ণ** : ফাংশনগুলো LLM-কে দেওয়া সিস্টেম মেসেজে অন্তর্ভুক্ত থাকে এবং এটি আপনার উপলব্ধ টোকেনের পরিমাণের মধ্যে গণনা করা হবে। 


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


**সংজ্ঞাসমূহ** 

`name` -  যে ফাংশনকে কল করতে চাই তার নাম। 

`description` - ফাংশনটি কীভাবে কাজ করে তার বিবরণ। এখানে স্পষ্ট এবং নির্দিষ্ট হওয়া গুরুত্বপূর্ণ। 

`parameters` - এমন মান এবং ফর্ম্যাটের একটি তালিকা যা আপনি চান মডেল তার উত্তরে তৈরি করুক। 


`type` - যেই ডেটা টাইপে প্রোপার্টিগুলি সংরক্ষিত হবে। 

`properties` - মডেল তার উত্তরে ব্যবহারের জন্য নির্দিষ্ট মানগুলির তালিকা। 


`name` - সেই প্রোপার্টির নাম যা মডেল তার ফরম্যাটকৃত উত্তরে ব্যবহার করবে। 

`type` - এই প্রোপার্টির ডেটা টাইপ। 

`description` - নির্দিষ্ট প্রোপার্টির বর্ণনা। 


**ঐচ্ছিক**

`required` - ফাংশন কল সম্পন্ন করার জন্য প্রয়োজনীয় প্রোপার্টি। 


### ফাংশন কল করা  
একটি ফাংশন সংজ্ঞায়িত করার পরে, এখন আমাদের এটি Chat Completion API কল-এ অন্তর্ভুক্ত করতে হবে। আমরা এটি করতে অনুরোধে `functions` যোগ করি। এই ক্ষেত্রে `functions=functions`।  

`function_call` কে `auto` সেট করার একটি অপশনও আছে। এর মানে হল ব্যবহারকারীর মেসেজের ভিত্তিতে কোন ফাংশন কল করা হবে তা LLM নিজেই সিদ্ধান্ত নেবে, আমরা নিজে তা নির্ধারণ করব না।  


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


এখন আসুন উত্তরটি দেখি এবং কিভাবে এটি বিন্যস্ত করা হয়েছে তা দেখি: 

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

আপনি দেখতে পারেন যে ফাংশনের নাম কল করা হয়েছে এবং ব্যবহারকারীর বার্তা থেকে, এলএলএম ফাংশনের আর্গুমেন্টগুলোর সাথে মেলানোর জন্য ডেটা খুঁজে পেয়েছে। 


## 3.একটি অ্যাপ্লিকেশনে ফাংশন কল একীকরণ।


LLM থেকে ফরম্যাট করা প্রতিক্রিয়া পরীক্ষা করার পরে, এখন আমরা এটি একটি অ্যাপ্লিকেশনে একীভূত করতে পারি।

### প্রবাহ পরিচালনা

এটি আমাদের অ্যাপ্লিকেশনে একীভূত করতে, আসুন নিম্নলিখিত ধাপগুলি গ্রহণ করি:

প্রথমে, আসুন Open AI সেবাগুলিতে কল করি এবং বার্তাটি `response_message` নামে একটি ভেরিয়েবলে সংরক্ষণ করি।


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


এখন আমরা সেই ফাংশনটি সংজ্ঞায়িত করব যা Microsoft Learn API কল করবে একটি কোর্সের তালিকা পেতে: 


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



একটি শ্রেষ্ঠ অনুশীলন হিসেবে, আমরা তখন দেখব মডেলটি একটি ফাংশন কল করতে চায় কিনা। তারপর, আমরা উপলব্ধ ফাংশনগুলির একটি তৈরি করব এবং সেটিকে কল করা ফাংশনের সাথে মিলিয়ে নেব। 
তারপর আমরা ফাংশনের আর্গুমেন্টগুলি নেব এবং সেগুলিকে LLM এর আর্গুমেন্টগুলির সাথে মানচিত্র করব।

অবশেষে, আমরা ফাংশন কল বার্তা এবং `search_courses` বার্তা দ্বারা ফিরিয়ে আনা মানগুলি সংযুক্ত করব। এটি LLM কে সমস্ত তথ্য দেয় যা এটি ব্যবহারকারীর কাছে প্রাকৃতিক ভাষায় উত্তর দিতে প্রয়োজন। 
উত্তর দিতে প্রাকৃতিক ভাষা ব্যবহার করে ব্যবহারকারীর কাছে সাড়া দেওয়ার জন্য প্রয়োজনীয় সব তথ্য প্রদান করে। 


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


এখন আমরা আপডেট হওয়া মেসেজটি LLM-এ পাঠাবো যাতে আমরা একটি API JSON ফরম্যাট করা উত্তর না নিয়ে একটি প্রাকৃতিক ভাষার উত্তর পেতে পারি। 


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## কোড চ্যালেঞ্জ

চমৎকার কাজ! Azure Open AI Function Calling এর শেখা চালিয়ে যাওয়ার জন্য আপনি তৈরি করতে পারেন: https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst 
 - ফাংশনের আরও প্যারামিটার যা শিক্ষার্থীদের আরও কোর্স খুঁজে পেতে সাহায্য করতে পারে। আপনি এখানে উপলব্ধ API প্যারামিটারগুলি খুঁজে পেতে পারেন: 
 - আরও তথ্য নেওয়ার জন্য আরেকটি ফাংশন কল তৈরি করুন, যেমন শিক্ষার্থীর নিজস্ব ভাষা 
 - ফাংশন কল এবং/অথবা API কল কোন উপযুক্ত কোর্স না ফেরালে এরর হ্যান্ডলিং তৈরি করুন 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
